# ProteomeXchangeから大腸がんプロテオミクスデータを取得する

**対応記事**: [article-02a-data-acquisition.md](../blog/article-02a-data-acquisition.md)  
**実行順序**: 2a番目  
**所要時間**: 約45分（ダウンロード時間除く）

---

## このNotebookで行うこと

Toyota et al. 2025 論文の公開データをProteomeXchangeからダウンロードし、データ構造を確認します。

- ProteomeXchangeの概要理解
- データセットPXD058672の概要確認
- ダウンロード方法とディレクトリ準備
- データ検証とファイル構造確認

## 前提条件

- [notebook_01_setup.ipynb](./notebook_01_setup.ipynb) が完了していること
- 約40GBのストレージ容量があること
- 安定したインターネット接続

## 1. 環境準備とディレクトリ作成

In [ ]:
import os       # ディレクトリ作成・パス操作用
import glob     # ファイルパターンマッチング用
import requests # データセット情報取得用
import pandas as pd  # データフレーム操作用
from pathlib import Path  # モダンなパス操作用

# プロジェクトルートディレクトリ設定
PROJECT_ROOT = Path("..").resolve()  # notebooksディレクトリから1つ上のレベル
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"  # 生データ保存ディレクトリ

print(f"プロジェクトルート: {PROJECT_ROOT}")
print(f"生データディレクトリ: {DATA_RAW_DIR}")

# データディレクトリ作成
DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)
print(f"✅ ディレクトリ作成完了: {DATA_RAW_DIR}")

## 2. ProteomeXchangeデータセット情報

### データセット概要

| リポジトリ | ID | 内容 |
|----------|-----|------|
| ProteomeXchange | **PXD058672** | RAWファイル, mzML |
| jPOST | **JPST003422** | DIA解析出力, 定量データ |

### 論文情報
- **著者**: Toyota N, Konno R, et al.
- **タイトル**: "Identification of Cancer-Associated Proteins in Colorectal Cancer Using Mass Spectrometry"
- **ジャーナル**: Proteomes 2025; 13(3):38
- **DOI**: https://doi.org/10.3390/proteomes13030038

In [ ]:
# データセット基本情報の定義
DATASET_INFO = {
    'proteomexchange_id': 'PXD058672',
    'jpost_id': 'JPST003422',
    'proteomexchange_url': 'https://proteomecentral.proteomexchange.org/cgi/GetDataset?ID=PXD058672',
    'jpost_url': 'https://repository.jpostdb.org/entry/JPST003422',
    'paper_doi': '10.3390/proteomes13030038',
    'total_size_gb': 38.74,
    'sample_count': 32,  # 16患者 × (Normal + Tumor)
    'instrument': 'Orbitrap Exploris 480'
}

print("=== データセット基本情報 ===")
for key, value in DATASET_INFO.items():
    print(f"{key}: {value}")

# 期待するファイル構造
EXPECTED_FILES = {
    'patients': 16,  # CRC01-CRC16
    'conditions': 2,  # Normal (N), Tumor (T) 
    'file_types': ['mzML', 'raw'],  # 主要なファイル形式
    'total_mzml_files': 32  # 16患者 × 2条件
}

print("\n=== 期待するファイル構造 ===")
for key, value in EXPECTED_FILES.items():
    print(f"{key}: {value}")

## 3. データダウンロード方法

### 手動ダウンロード（推奨）

**以下の手順をブラウザで実行してください:**

1. **ProteomeCentral**にアクセス: https://proteomecentral.proteomexchange.org
2. **「Datasets」**をクリック
3. Filter欄に **`PXD058672`** を入力して検索
4. **Toyota et al. (2025)**のデータセットをクリック
5. **「jPOST dataset」**のリンクをクリック
6. jPOSTページの **「Download all」**ボタンをクリック
7. ダウンロードしたファイルを `data/raw/` ディレクトリに展開

### 注意事項
- 総容量: 約40GB
- ダウンロード時間: 回線速度により数時間～半日
- 安定したネットワーク環境で実行してください

In [ ]:
# ダウンロード状況の確認
def check_download_status(data_dir):
    """データディレクトリの現在の状況をチェック"""
    
    print("=== ダウンロード状況確認 ===")
    
    # ディレクトリ存在確認
    if not data_dir.exists():
        print(f"❌ データディレクトリが存在しません: {data_dir}")
        return False
    
    # mzMLファイル検索
    mzml_files = list(data_dir.glob("**/*.mzML")) + list(data_dir.glob("**/*.mzml"))
    
    # RAWファイル検索
    raw_files = list(data_dir.glob("**/*.raw")) + list(data_dir.glob("**/*.RAW"))
    
    # zipファイル検索（ダウンロード中/未展開）
    zip_files = list(data_dir.glob("**/*.zip"))
    
    print(f"mzMLファイル数: {len(mzml_files)}")
    print(f"RAWファイル数: {len(raw_files)}")
    print(f"ZIPファイル数: {len(zip_files)}")
    
    # ファイル名パターンの確認
    if mzml_files:
        print("\n=== mzMLファイル例 ===")
        for i, file_path in enumerate(sorted(mzml_files)[:5]):  # 最初の5ファイル
            file_size_mb = file_path.stat().st_size / 1024**2  # ファイルサイズ（MB）
            print(f"{i+1:2d}. {file_path.name} ({file_size_mb:.1f} MB)")
        
        if len(mzml_files) > 5:
            print(f"    ... 他 {len(mzml_files) - 5} ファイル")
    
    # 期待する患者ID（CRC01-CRC16）の確認
    expected_patients = [f"CRC{i:02d}" for i in range(1, 17)]  # CRC01-CRC16
    found_patients = set()
    
    for file_path in mzml_files:
        file_name = file_path.name
        for patient_id in expected_patients:
            if patient_id in file_name:
                found_patients.add(patient_id)
    
    print(f"\n=== 患者IDの検出状況 ===")
    print(f"検出された患者数: {len(found_patients)} / {len(expected_patients)}")
    
    if found_patients:
        missing_patients = set(expected_patients) - found_patients
        if missing_patients:
            print(f"未検出の患者ID: {sorted(missing_patients)}")
        else:
            print("✅ 全患者IDが検出されました")
    
    # 総ファイル数の判定
    total_expected = EXPECTED_FILES['total_mzml_files']
    if len(mzml_files) == total_expected:
        print(f"\n🎉 ダウンロード完了! ({len(mzml_files)}/{total_expected} ファイル)")
        return True
    elif len(mzml_files) > 0:
        print(f"\n⚠️ 一部ダウンロード済み ({len(mzml_files)}/{total_expected} ファイル)")
        return False
    else:
        print(f"\n❌ mzMLファイルが見つかりません")
        if zip_files:
            print("ZIPファイルが見つかりました。展開してください。")
        else:
            print("データダウンロードから開始してください。")
        return False

# ダウンロード状況をチェック
download_complete = check_download_status(DATA_RAW_DIR)

## 4. ファイル構造の詳細確認

データが正しくダウンロードされた場合、詳細なファイル情報を表示します。

In [ ]:
def analyze_mzml_files(data_dir):
    """mzMLファイルの詳細解析"""
    
    # mzMLファイル一覧取得
    mzml_files = list(data_dir.glob("**/*.mzML")) + list(data_dir.glob("**/*.mzml"))
    
    if not mzml_files:
        print("❌ mzMLファイルが見つかりません")
        return None
    
    print("=== mzMLファイル詳細解析 ===")
    
    # ファイル情報をデータフレームで整理
    file_info_list = []
    
    for file_path in sorted(mzml_files):
        file_size_mb = file_path.stat().st_size / 1024**2  # MB単位
        file_name = file_path.name
        
        # 患者IDとサンプル種類の抽出
        patient_id = "Unknown"
        sample_type = "Unknown"
        
        # CRC01-N, CRC01-T 等のパターンをチェック
        for i in range(1, 17):  # CRC01-CRC16
            patient_pattern = f"CRC{i:02d}"
            if patient_pattern in file_name:
                patient_id = patient_pattern
                # Normal/Tumorの判定
                if "-N" in file_name or "_N" in file_name or "Normal" in file_name:
                    sample_type = "Normal"
                elif "-T" in file_name or "_T" in file_name or "Tumor" in file_name:
                    sample_type = "Tumor"
                break
        
        file_info_list.append({
            'ファイル名': file_name,
            '患者ID': patient_id,
            'サンプル種類': sample_type,
            'ファイルサイズ(MB)': round(file_size_mb, 1)
        })
    
    # データフレーム作成
    files_df = pd.DataFrame(file_info_list)
    
    # 基本統計
    total_size_gb = files_df['ファイルサイズ(MB)'].sum() / 1024
    normal_count = len(files_df[files_df['サンプル種類'] == 'Normal'])
    tumor_count = len(files_df[files_df['サンプル種類'] == 'Tumor'])
    unique_patients = files_df['患者ID'].nunique()
    
    print(f"総ファイル数: {len(files_df)}")
    print(f"総サイズ: {total_size_gb:.1f} GB")
    print(f"Normalサンプル: {normal_count}")
    print(f"Tumorサンプル: {tumor_count}")
    print(f"患者数: {unique_patients}")
    
    # ファイル一覧表示（最初の10ファイル）
    print("\n=== ファイル一覧（先頭10ファイル） ===")
    display(files_df.head(10))
    
    # 患者別ファイル数確認
    patient_summary = files_df.groupby(['患者ID', 'サンプル種類']).size().unstack(fill_value=0)
    print("\n=== 患者別ファイル数 ===")
    display(patient_summary)
    
    return files_df

# mzMLファイルが存在する場合のみ詳細解析実行
if download_complete:
    files_summary = analyze_mzml_files(DATA_RAW_DIR)
else:
    print("⚠️ ダウンロードが完了していません。データダウンロード後に再実行してください。")

## 5. 次のステップ準備

データダウンロードが完了したら、以下のファイルが利用可能になります。

In [ ]:
# 次のステップで使用するファイルパスの準備
def prepare_next_step_paths(data_dir):
    """次のステップ用のファイルパス情報を準備"""
    
    # 設定ファイル（次のステップで作成される予定）
    paths_config = {
        'data_raw_dir': str(data_dir),
        'mzml_pattern': str(data_dir / "**/*.mzML"),
        'fasta_dir': str(data_dir),  # FASTAファイル保存先
        'results_dir': str(PROJECT_ROOT / "results"),
        'sage_output_dir': str(PROJECT_ROOT / "results" / "sage")
    }
    
    print("=== 次のステップで使用するパス情報 ===")
    for key, path in paths_config.items():
        print(f"{key}: {path}")
    
    # 必要なディレクトリの作成
    results_dir = Path(paths_config['results_dir'])
    sage_dir = Path(paths_config['sage_output_dir'])
    
    results_dir.mkdir(parents=True, exist_ok=True)
    sage_dir.mkdir(parents=True, exist_ok=True)
    
    print("\n✅ 次のステップ用ディレクトリ作成完了")
    
    return paths_config

# パス情報準備
next_step_paths = prepare_next_step_paths(DATA_RAW_DIR)

# 簡易的なファイルカウント
mzml_count = len(list(DATA_RAW_DIR.glob("**/*.mzML"))) + len(list(DATA_RAW_DIR.glob("**/*.mzml")))
print(f"\n=== 準備完了確認 ===")
print(f"検出mzMLファイル数: {mzml_count}")
if mzml_count >= EXPECTED_FILES['total_mzml_files']:
    print("🎉 データ取得完了！次のステップ（データ形式解説）に進めます")
else:
    print(f"⚠️ 一部ファイル不足（期待: {EXPECTED_FILES['total_mzml_files']}, 実際: {mzml_count}）")

## トラブルシューティング

### 1. ダウンロードが途中で止まる
- ブラウザの設定で「大きなファイルのダウンロード継続」を有効にする
- 安定したネットワーク環境で再試行

### 2. ファイルが見つからない
- ダウンロードしたZIPファイルを `data/raw/` に展開したか確認
- ファイル名の大文字小文字（.mzML vs .mzml）を確認

### 3. 容量不足
- 約40GBの空き容量が必要
- 不要な mzMLファイルのみダウンロードする（RAWファイルは解析で使用しない）

## まとめ

✅ **完了項目**:
- ProteomeXchangeデータセット（PXD058672）の概要理解
- データダウンロード方法の確認
- ファイル構造の検証
- 次ステップ用ディレクトリの準備

**取得データ**:
- 16患者分 × 2条件（Normal/Tumor）= 32 mzMLファイル
- 総容量: 約40GB
- データセット: Toyota et al. 2025 DIA-MS大腸がんプロテオミクス

---

## Navigation

⬅️ **前回**: [notebook_01_setup.ipynb](./notebook_01_setup.ipynb) — 環境構築  
➡️ **次回**: [notebook_02b_data_formats.ipynb](./notebook_02b_data_formats.ipynb) — データ形式の理解

---

*このNotebookは [article-02a-data-acquisition.md](../blog/article-02a-data-acquisition.md) に対応しています。*